In [ ]:
import pickle
import matplotlib.pyplot as plt
import japanize_matplotlib
import numpy as np
from sklearn.svm import SVC
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import pandas as pd
import seaborn as sns

In [ ]:
# data_file_path = "../data/20250512/processed/ishii_merged.pkl"
data_file_path = Path("../../data/20250604/processed/20250604_1_ishii_ica_epochs.pkl")
# data_file_path = "../../data/bcidatasetIV2a/processed/merged.pkl"

In [ ]:
# pickle?EEG????????
with open(data_file_path, 'rb') as f:
    data = pickle.load(f)

epochs = data['epochs'] # ???????
epochs_list = epochs.get_data()  # ????????????????
labels = data['labels']  # ???
# drop_indices = data['drop_indices']  # ????????????????
fs = epochs.info['sfreq']  # ?????????

In [ ]:
ch_map = list(enumerate(epochs.info["ch_names"]))
ch_map

In [ ]:
use_chs = ["C5","C1","C2","C6","CP3","CP4"]
use_chs_idx = [epochs.info["ch_names"].index(ch) for ch in use_chs]
use_chs_idx

In [ ]:
# ?: axis=1?2, 4, 7??????????????
# selected_indices = [2, 4, 7]
# epochs_selected = epochs_list[:, selected_indices, :]
# print(f"????: {epochs_list.shape}")
# print(f"??????: {epochs_selected.shape}")

# ?????????????????
use_chs = ["C5","C1","C2","C6","CP3","CP4"]
use_chs_idx = [epochs.info["ch_names"].index(ch) for ch in use_chs]
use_chs_idx

In [ ]:
# labels?9999????????
valid_indices = np.where(labels != 9999)[0]
# epochs_list = epochs_list[valid_indices]
labels = labels[valid_indices]

In [ ]:
epochs_list.shape

In [ ]:
labels.shape

In [ ]:
# 1??????????DFT?????????????
def calc_spectrum(data, band_count, fs):
    n = len(data)
    freq = np.fft.fftfreq(n, 1/fs)
    spectrum = np.fft.fft(data)
    # 1Hz-40Hz???4Hz??????????????
    band_lower = 0
    band_upper = 40
    band_powers = [None] * band_count
    for i in range(band_count):
        band = band_lower+i*(band_upper-band_lower)/band_count, band_lower+(i+1)*(band_upper-band_lower)/band_count
        band_power = np.sum(np.abs(spectrum[(freq >= band[0]) & (freq < band[1])])**2)
        band_powers[i] = band_power
    
    # band_powers???None??????assert
    # assert all([v is not None for v in band_powers]), "Some band powers are None"
    assert all(band_powers), "Some band powers are None"
    return band_powers

In [ ]:
# ?????6???????????????FFT???????
def calc_windowed_spectrum(data, band_count, fs, window_count=6):
    n = len(data)
    window_size = n // window_count
    all_band_powers = []
    
    for i in range(window_count):
        # ??????????????????
        start_idx = i * window_size
        end_idx = (i + 1) * window_size if i < window_count - 1 else n
        
        # ???????????????
        window_data = data[start_idx:end_idx]
        
        # ?????????FFT???
        window_band_powers = calc_spectrum(window_data, band_count, fs)
        
        # ?????
        all_band_powers.extend(window_band_powers)
    
    return all_band_powers

In [ ]:
e0 = epochs_list[0, :, :]
e0c0 = epochs_list[0, 0, :]
e0c0

In [ ]:
calc_windowed_spectrum(e0c0, 10, fs, 6)

In [ ]:
# epochs_list[0]????????DFT???
def calc_spectrum_all_channels(epoch, band_count, fs, window_count=6):
    n_channels = epoch.shape[0]
    # ??????????band_count ? window_count ??????????????
    band_powers = np.zeros((n_channels, band_count * window_count))
    for i in range(n_channels):
        band_powers[i] = calc_windowed_spectrum(epoch[i, :], band_count, fs, window_count)
    return band_powers

In [ ]:
epochs_list

In [ ]:
from tqdm.notebook import tqdm
from joblib import Parallel, delayed
import multiprocessing

print(use_chs)
band_count = 10
window_count = 3

time_start = int(fs)
time_end = time_start+int(fs*3)

# ?????????(window_count??)
# all_epochs_band_powers = np.zeros((epochs_list.shape[0], epochs_list.shape[1], band_count * window_count))
all_epochs_band_powers = np.zeros((epochs_list.shape[0], 63, band_count * window_count))

# ????????????
def process_epoch(i):
    # return calc_spectrum_all_channels(epochs_list[i, use_chs_idx, :], band_count, fs, window_count)
    return calc_spectrum_all_channels(epochs_list[i, :, time_start:time_end], band_count, fs, window_count)

# ????CPU??????(-1???????)
n_jobs = multiprocessing.cpu_count()
print(f"????CPU???: {n_jobs}")

# tqdm???????????????????
results = Parallel(n_jobs=n_jobs, verbose=0)(
    delayed(process_epoch)(i) for i in tqdm(range(epochs_list.shape[0]), desc='???????')
)

# ????????
for i, result in enumerate(results):
    all_epochs_band_powers[i] = result

print(all_epochs_band_powers)
all_epochs_band_powers.shape

In [ ]:
# all_epochs_band_powers?pickle???
# data_file_path??????????????????????
output_file_path = data_file_path.parent / (data_file_path.stem + '_band_powers.pkl')
with open(output_file_path, 'wb') as f:
    pickle.dump({
        'band_powers': all_epochs_band_powers,
        'labels': labels,
        'fs': fs,
        'use_chs': use_chs,
        'use_chs_idx': use_chs_idx
    }, f)
print(f"??????????: {output_file_path}")

In [ ]:
# ????2???????(?????????)
# ?????????10???????? ? 6???????
# ?????????? ? 10 ? 6 ?????????????
X = all_epochs_band_powers.reshape(all_epochs_band_powers.shape[0], -1)
print(f"????shape: {X.shape}")
print(f"????shape: {labels.shape}")

# ???????????????????
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.3, random_state=42, stratify=labels)

In [ ]:
# ??????????
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# SVM?????????
svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm.fit(X_train_scaled, y_train)

# ?????????
y_pred = svm.predict(X_train_scaled)

# ??
print("??????:")
# print(classification_report(y_test, y_pred))# ?????????
print(classification_report(y_train, y_pred))# ?????????
unique_labels, counts = np.unique(labels, return_counts=True)
print("??????:")
for label, count in zip(unique_labels, counts):
    print(f"??? {label}: {count}?")

In [ ]:
# ???????
cm = confusion_matrix(y_train, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
           xticklabels=unique_labels, 
           yticklabels=unique_labels)
plt.xlabel('?????')
plt.ylabel('??????')
plt.title('????')
plt.show()

In [ ]:
# ?????????????
cv_scores = cross_val_score(svm, X_train_scaled, y_train, cv=10)
print(f"?????????????: {cv_scores}")
print(f"????: {cv_scores.mean():.4f} ? {cv_scores.std():.4f}")
# ??????????????????
plt.figure(figsize=(8, 6))
plt.plot(cv_scores, marker='o', linestyle='-', color='b')
plt.title('?????????????')
plt.xlabel('Fold')
plt.ylabel('???')
plt.xticks(range(len(cv_scores)))
plt.grid()
plt.show()

In [ ]:
# ???????????????
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.1, 0.01, 0.001],
    'kernel': ['rbf', 'linear', 'poly']
}

grid_search = GridSearchCV(SVC(random_state=42), param_grid, cv=10, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

print(f"???????: {grid_search.best_params_}")
print(f"?????: {grid_search.best_score_:.4f}")


In [ ]:
# ?????????
best_svm = grid_search.best_estimator_
y_pred_best = best_svm.predict(X_test_scaled)

# ???????CV??????
best_cv_scores = cross_val_score(best_svm, X_train_scaled, y_train, cv=10)
print(f"???????????????????: {best_cv_scores}")
print(f"??????????: {best_cv_scores.mean():.4f} ? {best_cv_scores.std():.4f}")

print("????????????:")
print(classification_report(y_test, y_pred_best))

# ???????
cm_best = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Blues',
           xticklabels=unique_labels, 
           yticklabels=unique_labels)
plt.xlabel('?????')
plt.ylabel('??????')
plt.title('??????????')
plt.show()